# Fluency-SpAM: data cleaning

Cleans the data saved by the experiment (`../Experiment`, Streamlit + MongoDB). Outputs are written next to this notebook:

1. **Fetch and anonymise.** Every participant document is read from MongoDB and saved to `raw/mongo_snapshot.json`; `raw/` is gitignored. Names and roll numbers are removed and each participant gets an ID (`P001`, `P002`, …). The result is `responses_anonymised.json`, and the ID ↔ identity key is `raw/participant_key.csv`. If `raw/` is missing, the committed anonymised data is used instead.
2. **Clean** → `cleaned_responses.csv`: one row per participant, word lists stored as JSON. The `Name` column holds the participant ID.
3. **Exclude** the participants listed in `EXCLUDED` → `cleaned_split_responses.csv`.
4. **Unique words** per column → `uniq_words.txt`, to help extend the transliteration key.
5. **Transliterate** romanised words to Devanagari with `transliteration_key.json` → `transliterated_split.csv` and `transliterated_data.pkl` (read by `../Analyses`). If any word is missing from the key, this step lists the missing words by category and stops.

In [ ]:
import csv
import json
import os
import pickle
from copy import deepcopy
from pathlib import Path

RAW_DIR = Path("raw")  # identifiable data, gitignored
RAW_SNAPSHOT = RAW_DIR / "mongo_snapshot.json"
PARTICIPANT_KEY = RAW_DIR / "participant_key.csv"  # participant ID <-> name / roll number
TRANSLITERATION_KEY = Path("transliteration_key.json")
MONGO_URI_FILE = Path("../Experiment/uri")  # used when $MONGO_URI is not set

FETCH_FROM_MONGO = True  # False: reuse the saved snapshot in raw/

# Participants dropped between cleaned_responses.csv and cleaned_split_responses.csv,
# e.g. test runs or repeat submissions: {"P001": "reason", ...}
EXCLUDED = {}

## Anonymisation helpers

IDs are stable: an existing entry in `raw/participant_key.csv` is reused, and new participants get the next free number. After anonymising, every name and roll number from the raw data is searched for in the output, and the step fails if any is still present.

In [ ]:
KEY_FIELDS = ["participant_id", "source_key", "name", "roll_number"]


def load_participant_key():
    if not PARTICIPANT_KEY.exists():
        return []
    with open(PARTICIPANT_KEY, "r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def save_participant_key(entries):
    PARTICIPANT_KEY.parent.mkdir(parents=True, exist_ok=True)
    with open(PARTICIPANT_KEY, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=KEY_FIELDS)
        writer.writeheader()
        writer.writerows(entries)


def assign_ids(people, width=3):
    """people: [(source_key, name, roll_number), ...] in a stable order.
    Returns {source_key: participant_id} and records new participants in the key file."""
    entries = load_participant_key()
    existing = {e["source_key"]: e["participant_id"] for e in entries}
    next_number = max((int(pid[1:]) for pid in existing.values()), default=0) + 1
    for source_key, name, roll_number in people:
        if source_key not in existing:
            existing[source_key] = f"P{next_number:0{width}d}"
            entries.append({"participant_id": existing[source_key], "source_key": source_key,
                            "name": name, "roll_number": roll_number})
            next_number += 1
    save_participant_key(entries)
    return {source_key: existing[source_key] for source_key, _, _ in people}


def check_no_identifiers(anonymised, identifiers):
    """Fails if any identifier survives anywhere in the anonymised data."""
    text = json.dumps(anonymised, ensure_ascii=False).lower()
    leaks = [s for s in identifiers if s and s.strip().lower() in text]
    if leaks:
        raise ValueError(f"{len(leaks)} names / roll numbers are still present after anonymisation")
    print(f"Checked {len(identifiers)} names / roll numbers: none present in the anonymised data.")

## 1. Fetch and anonymise

Reads every document in `semantic_fluency_db.participants` (read-only), oldest submission first. The connection string comes from `$MONGO_URI`, or from the first non-empty line of `../Experiment/uri`. If the fetch fails, the saved snapshot is used instead.

`name` and `rno` are removed and `participant_id` is added. The step also reports roll numbers that occur more than once, by ID, so repeat or test submissions can be added to `EXCLUDED`; `raw/participant_key.csv` shows who each ID is.

In [ ]:
DB_NAME = "semantic_fluency_db"
COLLECTION = "participants"


def get_mongo_uri():
    uri = os.environ.get("MONGO_URI", "").strip()
    if uri:
        return uri
    with open(MONGO_URI_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                return line.strip()
    raise RuntimeError(f"No MongoDB URI: set $MONGO_URI or put it in {MONGO_URI_FILE}")


def fetch_from_mongo(uri):
    """Returns every participant document, oldest submission first, as plain JSON
    (ObjectId -> {"$oid": ...})."""
    from bson import json_util
    from pymongo import MongoClient

    client = MongoClient(uri, serverSelectionTimeoutMS=30000)
    try:
        docs = list(client[DB_NAME][COLLECTION].find({}).sort([("submitted_at", 1), ("_id", 1)]))
    finally:
        client.close()
    return json.loads(json_util.dumps(docs))


def anonymise(raw_path, out_path):
    with open(raw_path, "r", encoding="utf-8") as f:
        docs = json.load(f)
    people = [(doc["_id"]["$oid"], doc.get("name", ""), str(doc.get("rno", ""))) for doc in docs]
    ids = assign_ids(people)

    by_roll = {}
    for source_key, _, roll in people:
        by_roll.setdefault(roll.strip().lower(), []).append(ids[source_key])
    for pids in by_roll.values():
        if len(pids) > 1:
            print(f"Same roll number submitted more than once: {pids}")

    anonymised = []
    for doc in docs:
        rest = {k: v for k, v in deepcopy(doc).items() if k not in ("name", "rno")}
        anonymised.append({"participant_id": ids[doc["_id"]["$oid"]], **rest})

    identifiers = {name for _, name, _ in people} | {roll for _, _, roll in people}
    check_no_identifiers(anonymised, identifiers)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(anonymised, f, ensure_ascii=False, indent=2)
    print(f"{out_path}: {len(anonymised)} participants")


if FETCH_FROM_MONGO:
    try:
        snapshot = fetch_from_mongo(get_mongo_uri())
        RAW_SNAPSHOT.parent.mkdir(parents=True, exist_ok=True)
        with open(RAW_SNAPSHOT, "w", encoding="utf-8") as f:
            json.dump(snapshot, f, ensure_ascii=False, indent=2)
        print(f"Fetched {len(snapshot)} documents into {RAW_SNAPSHOT}")
    except Exception as exc:
        print(f"WARNING: could not fetch from MongoDB ({type(exc).__name__}). Using the saved snapshot if there is one.")

if RAW_SNAPSHOT.exists():
    anonymise(RAW_SNAPSHOT, Path("responses_anonymised.json"))
else:
    print(f"{RAW_SNAPSHOT} not found: using the committed responses_anonymised.json")

## 2. Clean

One row per participant; word lists are JSON strings.

* **VFT times.** The app records seconds since the word box appeared, timed in the participant's browser (older documents were timed on the server). The `… VFT Words with Response Times` columns store milliseconds: the first response's time, then the interval between consecutive responses, rounded to 2 decimals. The recorded seconds are kept in `… VFT Raw Times (s since start)`.
* **SpAM coordinates** are stored as `[word, x, y]` in the order the words were shown, as recorded by the app: the centre of each word box, normalised by the board size, with y measured from the bottom. The board is square, so x and y share one scale; its size in pixels is in `… SpAM Plane Size (px)`. Older documents have no board size and came from a non-square board, so their x and y scales differ.
* **Ratings.** Comfort answers ("Most Uncomfortable" … "Most Comfortable") become `1`–`5`. Language proficiency and acquisition order ("1 - Least Proficient", "1 - First") become their digit, stored as `[language, proficiency, order]`.
* **Strategies** are a JSON object keyed by category, since the app asks once per category. Documents saved before the "Marketplace" question was removed also have a `Marketplace` answer.
* `Category Order` is the randomised order the participant saw the categories in.

In [ ]:
# Category key in the app -> column label
CATEGORIES = {
    "animals": "Animal",
    "body_parts": "Body Parts",
    "fruitsnveg": "Fruits and Vegetables",
}
# Order of the free-text answers in p["strats"] (exit_poll_1 in app.py); older documents have a 4th, Marketplace
STRATEGY_LABELS = ["Animal", "Body Parts", "Fruits and Vegetables", "Marketplace"]
COMFORT = {
    "Most Uncomfortable": "1",
    "Moderately Uncomfortable": "2",
    "Neutral": "3",
    "Moderately Comfortable": "4",
    "Most Comfortable": "5",
}

BASE_COLUMNS = ["Name", "Hindi Read", "Hindi Write", "Eng Read", "Eng Write", "Strategies",
                "Langs with Conf and Acq", "State/UT", "Age", "Gender", "Education",
                "Dominant Hand", "Alert_Time", "Other Info"]


def vft_column(label):
    return f"{label} VFT Words with Response Times"


def spam_column(label):
    return f"{label} SpAM Words with Normalised Coords"


def raw_times_column(label):
    return f"{label} VFT Raw Times (s since start)"


def plane_column(label):
    return f"{label} SpAM Plane Size (px)"


def header():
    cols = list(BASE_COLUMNS)
    for label in CATEGORIES.values():
        cols += [vft_column(label), spam_column(label)]
    cols += ["First Language", "Category Order", "Submitted At"]
    cols += [raw_times_column(label) for label in CATEGORIES.values()]
    cols += [plane_column(label) for label in CATEGORIES.values()]
    return cols


def _na(value):
    return "N/A" if value is None or value == "" else value


def _digit(value):
    """'1 - Least Proficient' -> '1', '3' -> '3', missing -> 'N/A'."""
    if value is None or str(value).strip() == "":
        return "N/A"
    return str(value).split("-")[0].strip()


def to_interval_ms(seconds_since_start):
    """Seconds since the category started -> ms of the first response, then ms between
    consecutive responses (2 d.p.)."""
    out, prev = [], 0.0
    for t in seconds_since_start:
        out.append(round((float(t) - prev) * 1000, 2))
        prev = float(t)
    return out


def clean_document(doc, name):
    """Returns one CSV row (dict keyed by header()) for a participant document."""
    row = {
        "Name": name,
        "Hindi Read": COMFORT.get(doc.get("hi_r"), _na(doc.get("hi_r"))),
        "Hindi Write": COMFORT.get(doc.get("hi_w"), _na(doc.get("hi_w"))),
        "Eng Read": COMFORT.get(doc.get("en_r"), _na(doc.get("en_r"))),
        "Eng Write": COMFORT.get(doc.get("en_w"), _na(doc.get("en_w"))),
        "Strategies": "N/A",
        "Langs with Conf and Acq": "N/A",
        "State/UT": _na(doc.get("location")),
        "Age": _na(doc.get("age")),
        "Gender": _na(doc.get("gender")),
        "Education": _na(doc.get("edu")),
        "Dominant Hand": _na(doc.get("dom_hand")),
        "Alert_Time": _na(doc.get("alert_time")),
        "Other Info": _na(doc.get("extra_info")),
        "First Language": _na(doc.get("first_lang")),
        "Submitted At": _na(doc.get("submitted_at")),
    }

    strats = doc.get("strats")
    if isinstance(strats, list):
        row["Strategies"] = json.dumps(dict(zip(STRATEGY_LABELS, strats)), ensure_ascii=False)

    lang_prof = doc.get("lang_prof") or {}
    lang_order = doc.get("lang_order") or {}
    langs = [[lang.lower(), _digit(lang_prof.get(lang)), _digit(lang_order.get(lang))]
             for lang in doc.get("lang_list") or []]
    if langs:
        row["Langs with Conf and Acq"] = json.dumps(langs, ensure_ascii=False)

    categories = doc.get("categories") or {}
    row["Category Order"] = json.dumps([CATEGORIES.get(c, c) for c in categories], ensure_ascii=False)
    unknown = [c for c in categories if c not in CATEGORIES]
    if unknown:
        print(f"WARNING {name}: categories {unknown} are not in CATEGORIES and are left out")

    for key, label in CATEGORIES.items():
        cat = categories.get(key)
        if cat is None:
            for col in (vft_column(label), spam_column(label), raw_times_column(label), plane_column(label)):
                row[col] = "N/A"
            continue
        words_and_rts = cat.get("words_and_rts") or []
        words = [w for w, _ in words_and_rts]
        seconds = [t for _, t in words_and_rts]
        coords = cat.get("words_and_coords") or {}

        missing = [w for w in words if w not in coords]
        extra = [w for w in coords if w not in words]
        if coords and (missing or extra):
            print(f"WARNING {name} / {label}: {len(missing)} VFT words without SpAM coords, "
                  f"{len(extra)} SpAM words not in the VFT list")

        row[vft_column(label)] = json.dumps(list(zip(words, to_interval_ms(seconds))), ensure_ascii=False)
        # VFT order, i.e. the order the words were presented on the SpAM plane
        spam = [[w, coords[w][0], coords[w][1]] for w in words if w in coords]
        spam += [[w, coords[w][0], coords[w][1]] for w in extra]
        row[spam_column(label)] = json.dumps(spam, ensure_ascii=False)
        row[raw_times_column(label)] = json.dumps(seconds)
        row[plane_column(label)] = json.dumps(cat["plane_px"]) if cat.get("plane_px") else "N/A"
    return row


def clean(in_path, out_path):
    with open(in_path, "r", encoding="utf-8") as f:
        docs = json.load(f)
    rows = [clean_document(doc, doc["participant_id"]) for doc in docs]
    with open(out_path, "w+", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        cols = header()
        writer.writerow(cols)
        for row in rows:
            writer.writerow([row[c] for c in cols])
    print(f"Wrote {len(rows)} participants to {out_path}")


clean("responses_anonymised.json", "cleaned_responses.csv")

## 3–5. Exclude, list unique words, transliterate

Words are lower-cased and stripped of non-alphanumeric characters before they are looked up in `transliteration_key.json`. If any are missing, the last step lists them by category and stops: add them to the key and re-run this cell.

In [ ]:
def columns_to_process():
    cols = []
    for label in CATEGORIES.values():
        cols.extend([vft_column(label), spam_column(label)])
    return cols


def apply_exclusions(in_csv, out_csv, excluded):
    with open(in_csv, "r", encoding="utf-8", newline="") as f:
        rows = list(csv.reader(f))
    header_row, body = rows[0], rows[1:]
    present = {row[0] for row in body}
    for pid in excluded:
        if pid not in present:
            print(f"WARNING: excluded participant {pid} is not in {in_csv}")
    kept = [row for row in body if row[0] not in excluded]
    with open(out_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, lineterminator="\n")
        writer.writerow(header_row)
        writer.writerows(kept)
    print(f"{out_csv}: kept {len(kept)} of {len(body)} participants, excluded {sorted(excluded) or 'none'}")


def _strip_special_chars(s: str) -> str:
    """Return lowered 's' with all non-alphanumeric Unicode characters removed."""
    if not isinstance(s, str):
        return s
    return str(''.join(ch for ch in s if ch.isalnum())).lower()


def write_unique_words(in_csv, out_txt):
    uniq_words = {}
    with open(in_csv, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if not row.get("Name"):
                break
            for col in columns_to_process():
                words = uniq_words.setdefault(col, set())
                if row.get(col) and row[col] != "N/A":
                    for item in json.loads(row[col]):
                        cleaned_word = _strip_special_chars(item[0])
                        if cleaned_word:
                            words.add(cleaned_word)
    with open(out_txt, "w", encoding="utf-8") as g:
        for col, words in uniq_words.items():
            g.write(f"{col}:\n")
            for word in sorted(words):
                g.write(f"\t{word}\n")
    print(f"Wrote {out_txt}")


def find_untransliterated(in_csv, trans):
    """Returns {column: sorted words missing from the transliteration key}."""
    missing = {}
    with open(in_csv, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if not row.get("Name"):
                break
            for col in columns_to_process():
                if row.get(col) and row[col] != "N/A":
                    for item in json.loads(row[col]):
                        cleaned_word = _strip_special_chars(item[0])
                        if cleaned_word and cleaned_word not in trans:
                            missing.setdefault(col, set()).add(cleaned_word)
    return {col: sorted(words) for col, words in missing.items()}


def transliterate(in_csv, out_csv, out_pkl, trans):
    missing = find_untransliterated(in_csv, trans)
    if missing:
        by_category = {}
        for col, words in missing.items():
            label = col.replace(" VFT Words with Response Times", "").replace(" SpAM Words with Normalised Coords", "")
            by_category.setdefault(label, set()).update(words)
        lines = [f"{label}:\n\t" + "\n\t".join(sorted(words)) for label, words in by_category.items()]
        n = len({w for words in missing.values() for w in words})
        raise LookupError(f"{n} words are not in {TRANSLITERATION_KEY}. Add them and re-run this cell.\n\n"
                          + "\n".join(lines))

    with open(in_csv, "r", encoding="utf-8") as f:
        data = csv.DictReader(f)
        fieldnames = data.fieldnames or []
        l = []
        csv_rows = []
        for row in data:
            if not row.get("Name"):
                break
            new_row = deepcopy(row)
            for col in columns_to_process():
                if new_row.get(col) and new_row[col] != "N/A":
                    lst = json.loads(new_row[col])
                    for i in range(len(lst)):
                        cleaned_word = _strip_special_chars(lst[i][0])
                        if cleaned_word:
                            lst[i][0] = trans[cleaned_word]
                    new_row[col] = lst
            l.append(new_row)
            csv_row = {}
            for key in fieldnames:
                value = new_row.get(key, "")
                if isinstance(value, (list, dict)):
                    csv_row[key] = json.dumps(value, ensure_ascii=False)
                else:
                    csv_row[key] = value
            csv_rows.append(csv_row)

    with open(out_pkl, "wb") as f:
        pickle.dump(l, f)
    print(f"Saved transliterated data to {out_pkl}")

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(csv_rows)
    print(f"Saved transliterated CSV data to {out_csv}")


with open(TRANSLITERATION_KEY, "r", encoding="utf-8") as f:
    trans = json.load(f)
print(f"Transliteration key: {len(trans)} entries")

apply_exclusions("cleaned_responses.csv", "cleaned_split_responses.csv", EXCLUDED)
write_unique_words("cleaned_split_responses.csv", "uniq_words.txt")
transliterate("cleaned_split_responses.csv", "transliterated_split.csv", "transliterated_data.pkl", trans)

## Appendix: automatic transliteration

Exploratory comparison, not part of the pipeline: romanised animal names converted to Devanagari with Aksharamukha and indic-transliteration, then to IPA with Epitran.

In [ ]:
import pandas as pd
import epitran

# Method 1: Aksharamukha
# aksharamukha 2.3 does `from ast import Str` (unused), and ast.Str was removed in Python 3.14
import ast
if not hasattr(ast, "Str"):
    ast.Str = ast.Constant
from aksharamukha import transliterate as akshara_trans

# Method 2: indic-transliteration
from indic_transliteration.sanscript import transliterate as indic_trans
from indic_transliteration.sanscript import HK, DEVANAGARI

# IPA converter
epi = epitran.Epitran('hin-Deva')


# ---------- Transliteration Functions ----------

def aksharamukha_deva(word):
    try:
        return akshara_trans.process("ISO", "Devanagari", word)
    except:
        return "ERROR"

def indic_deva(word):
    try:
        return indic_trans(word, HK, DEVANAGARI)
    except:
        return "ERROR"

def to_ipa(deva_word):
    try:
        return epi.transliterate(deva_word)
    except:
        return "ERROR"


# ---------- Your word list ----------
animals = [
    "kutta",
    "billi",
    "gai",
    "ghoda",
    "bakri",
    "sher",
    "hathi",
    "bandar",
    "machli",
    "oont",
    "gadha",
    "lomdi",
    "bhalu",
    "murga",
    "tota",
    "chuha",
    "mendak",
    "saanp",
    "mor",
    "kabootar"
]



# ---------- Build comparison table ----------
rows = []

for w in animals:
    deva_ak = aksharamukha_deva(w)
    deva_ind = indic_deva(w)

    ipa_ak = to_ipa(deva_ak) if deva_ak != "ERROR" else "ERROR"
    ipa_ind = to_ipa(deva_ind) if deva_ind != "ERROR" else "ERROR"

    rows.append([w, deva_ak, ipa_ak, deva_ind, ipa_ind])

df = pd.DataFrame(
    rows,
    columns=[
        "Roman Input",
        "Aksharamukha Devanagari",
        "Aksharamukha IPA",
        "Indic-Translit Devanagari",
        "Indic-Translit IPA",
    ]
)

print(df)